<a href="https://colab.research.google.com/github/Shemuel-bot/AI-ball-notebooks/blob/main/gpt_ball_trap_brain_single_head_attention_fixed_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

ai_ball_7_.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1RScD6nl_1_teIc-eGZ_Tao-A5BP6xo1J

# AI Ball — Ball Trap Brain

A toy "ball trap" symbolic model: input characters are tokenized against an alphabet, turned into a ball trajectory through a ring of deflectors, and the deflectors the ball falls through ("holes") produce the output characters.

This notebook contains:
1. The tokenizer and the `Ball_Trap_Brain` model itself
2. Fitness evaluation utilities
3. A genetic algorithm (`main`) that evolves a single brain
4. A `Council_Brain` that combines several independently-trained brains by majority vote
5. A **continuous relaxation** training method (`Ball_Trap_Brain.train_continuous`) that trains a brain with gradient descent instead of the genetic algorithm, and can optionally keep the trained continuous parameters usable directly (`predict_continuous`, `evaluate_continuous`) instead of only ever discretizing them back into the original discrete brain
6. An optional, guarded cell that reproduces the original end-to-end council training run

## Imports & tokenizer


In [ ]:

import random
import math
import copy

def TokenizeInput(input, alphabet):
    input_str = str(input)
    matches = []

    for order, item in enumerate(alphabet):
        item_str = str(item)
        if not item_str:
            continue

        # Loop to find ALL occurrences of the item instead of just the first
        start_pos = 0
        while True:
            index = input_str.find(item_str, start_pos)
            if index == -1:
                break

            matches.append((index, -len(item_str), order, item))

            # Move start position forward by 1 to continue searching the rest of the string
            # (Adding 1 instead of len(item_str) allows it to safely catch overlapping patterns)
            start_pos = index + 1

    # Sorts by starting index, then by longest string, then by original order
    matches.sort(key=lambda match: (match[0], match[1], match[2]))

    non_overlapping_items = []
    last_end_index = 0

    for match in matches:
        start_index = match[0]
        length = -match[1]  # The length was stored as a negative number
        item = match[3]

        # Only add the token if it doesn't overlap with the previously added token
        if start_index >= last_end_index:
            non_overlapping_items.append(item)
            last_end_index = start_index + length
    #print(non_overlapping_items)
    return non_overlapping_items

## `Ball_Trap_Brain`

Includes the original genetic-algorithm-oriented methods (`clone`, `Mutate`, `shift_alphabet`, ...) plus a gradient-based continuous relaxation trainer, split across a few methods:
- `train_continuous(...)`: trains the continuous parameters with gradient descent (see its docstring for details on the relaxation scheme). Temperature cools from `temp_start` to `temp_end` (linear) and `index_temp` cools from `index_temp_start` to `index_temp_end` (power-`index_temp_power` schedule) over `epochs`, with the learning rate cosine-decaying alongside them and gradients clipped each step for stability; by default the best-loss checkpoint seen during training is restored at the end rather than whatever the final epoch produced. Returns `(loss_history, accuracy_history)`.
- `predict_continuous(input_str, ...)`: answer a question using the **continuous** brain directly, without discretizing anything -- no dependency on `deflectors_list` etc at all.
- `evaluate_continuous(test_cases, ...)`: like `evaluate_performance`, but scores `predict_continuous` instead of `compute_input`.
- `commit_continuous()`: discretize already-trained continuous parameters into the discrete brain without training any further.

**Two pollution bugs, now fixed, that affected discrete *and* continuous output alike:**
1. *Leading-position leakage.* The loss only scored a right-aligned tail of each trajectory; leading positions outside that window had no training signal at all and could leak arbitrary symbols into the real output. Fixed by folding "should this position emit at all" into the *same* per-position loss as a genuine "emit nothing" class -- `never_fired = 1 - out_dist.sum()` already forms a valid (K+1)-way distribution with the K real symbols, so every position (aligned or not) now has a well-defined target under one unified objective, instead of a second, separately-weighted penalty term that destabilized training by fighting the main loss over shared deflectors.
2. *Repeat/self-loop cycling.* Once a trajectory hit a hole, the "first hole reached" objective stopped caring what happened for the rest of its `cap` steps -- so a hole deflector could cycle with another hole (or even itself) and just re-emit symbols for the whole `cap` budget (e.g. `"3333333333"`, `"2323232323"`). Fixed with `repeat_penalty_weight` (default 0.3), which penalizes the expected number of *additional* hole-hits after the first one per trajectory -- computed from the same batched forward pass as the main loss, so it targets whatever cycle each specific trajectory is actually at risk of.

**Blank-output collapse on long inputs, and `blank_loss_weight`.** Long inputs have many more blank-target positions (leading positions with no aligned target character, per bug #1 above) than fire-target ones -- a 500-character input producing a 1-character output has 499 blank targets and only 1 real one. Averaging the per-position loss flatly over all L positions (the naive approach) lets that single fire position's gradient get diluted to roughly 1/L of the total loss mass, while "always predict blank" already captures nearly all of the flat loss for free -- so training can converge to a real local minimum where every trajectory predicts blank and the model's output is empty regardless of input. To prevent this, the fire-target and blank-target losses are normalized *separately* (mean within each group) before being combined as `fire_loss_mean + blank_loss_weight * blank_loss_mean`, so one fire position's contribution stays comparable in scale to the average blank position's however many blank positions surround it. `blank_loss_weight` (default 1.0) then controls their relative importance directly -- lower it (e.g. 0.1-0.3) if you want firing at the right position prioritized even more strongly over suppressing leading positions. The printed per-unit stats now also report `fire_acc` (accuracy restricted to positions that should actually fire) separately from the overall `token_acc` (which is dominated by the easy, mostly-correct blank predictions and can look deceptively high -- e.g. ~0.99 -- even while the model has completely collapsed to never firing).

**`commit` flag:** `train_continuous(..., commit=True)` (the default) discretizes the trained parameters back into this instance's own `deflectors_list`, `alphabet_directions` and `alphabet_start_positions` -- so every other method (`compute_input`, `ricochet`, `clone`, `Mutate`, ...) keeps working exactly as before, just with gradient-trained values. Pass `commit=False` to leave the discrete brain completely untouched and only keep the trained continuous version around for `predict_continuous`/`evaluate_continuous` -- useful for comparing continuous vs. discrete behavior, or for iterating with more `train_continuous` calls before ever committing to a discrete brain (call `commit_continuous()` whenever you're ready to).

**How `token_acc` is scored (printed during training):** within each epoch's batch, for every trajectory position (aligned *or* blank), it checks whether the predicted class -- top symbol if emit-probability >= 0.5, else "blank" -- matches the target class, and reports the fraction that do. It's read off the same numbers already used for the loss -- no extra simulation -- but it's still a *character-position* metric, not a *question* metric: it doesn't check whether a whole answer matches end to end, and it's computed on the soft relaxation, not the real discrete brain. For real, whole-question exact-match accuracy, use `evaluate_performance` (discrete) or `evaluate_continuous` (continuous).

### Single-head attention
The discrete path uses deterministic leave-one-out attention over input tokens. The continuous path uses the same mask with learned Q/K/V projections and a bounded attention gain.


In [ ]:

class Ball_Trap_Brain:
    def __init__(self, deflectors, bounce, alphabet, output, holes_multiplier=1, direction_range=10,
                 attention_enabled=False, attention_gain=1.0):
        self.output = output
        # Attention controls. Continuous training learns Q/K/V projections. The
        # discrete brain uses that learned head only after commit_continuous(); until
        # then its original deterministic travel rule is preserved.
        self.attention_enabled = attention_enabled
        self.attention_gain = attention_gain
        self.holes_multiplier = holes_multiplier
        self.ball_energy = bounce*2
        self.alphabet = alphabet
        # Each alphabet token gets its own integer direction in
        # [-direction_range, direction_range] instead of only +-1. A wider range
        # gives travel()'s leave-one-out, position-weighted sum far more
        # resolution to separate tokens by -- with only +-1, two different tokens
        # were indistinguishable to the sum beyond their single bit of sign, which
        # was a capacity bottleneck independent of (and compounding) the
        # order-invariance one that position weighting alone fixes.
        self.direction_range = direction_range
        self.alphabet_directions = self.choose_alphabet_direction(len(alphabet))
        self.alphabet_start_positions = self.start_positions(len(alphabet), deflectors)
        self.deflectors_list = [[random.randint(0, deflectors-1), bounce] for i in range(deflectors)]
        self.set_holes(output, deflectors, self.holes_multiplier)

    def set_holes (self, output, deflectors, holes_multiplier):
        for i in range(len(output)):
          for j in range(holes_multiplier):
            self.deflectors_list[random.randint(0, deflectors-1)][1] = output[i]

    def choose_alphabet_direction(self, alphabet):
        return [random.randint(-self.direction_range, self.direction_range) for _ in range(alphabet)]

    def start_positions(self, alphabet, deflectors):
        return [random.randint(0, deflectors-1) for i in range(alphabet)]


    def _single_head_attention_discrete(self, directions, starts, index):
        """Deterministic counterpart of the *trained* continuous Q/K/V head.

        The previous version used a hand-written score rule that was unrelated to
        the Q/K/V projections learned by the continuous brain. That made
        ``commit_continuous()`` inherently lossy: the continuous model could be
        accurate while the committed discrete model was solving a different
        function.  When continuous Q/K/V weights are available, use those exact
        weights here with hard (discrete) token directions and start positions.
        """
        if not self.attention_enabled or len(directions) <= 1:
            return 0

        # Preferred path: use the Q/K/V parameters learned by the continuous brain.
        if hasattr(self, '_discrete_q_proj'):
            q_proj = self._discrete_q_proj
            k_proj = self._discrete_k_proj
            v_proj = self._discrete_v_proj
            gain = self._discrete_attn_gain
            dim = len(q_proj[0]) if q_proj and q_proj[0] else 1
            ds = max(1.0, float(self.direction_range))
            D = max(2, len(self.deflectors_list))
            ps = max(1.0, float(D - 1))

            features = []
            for d, p in zip(directions, starts):
                features.append([float(d) / ds, float(p) / ps, 1.0])

            def proj(x, w):
                return [sum(x[r] * w[r][c] for r in range(3)) for c in range(len(w[0]))]

            q_all = [proj(features[j], q_proj) for j in range(len(features))]
            k_all = [proj(features[j], k_proj) for j in range(len(features))]
            v_all = [sum(features[j][r] * v_proj[r][0] for r in range(3)) for j in range(len(features))]

            scores = []
            values = []
            q = q_all[index]
            for j in range(len(features)):
                if j == index:
                    continue
                score = sum(q[c] * k_all[j][c] for c in range(dim)) / math.sqrt(float(dim))
                scores.append(score)
                values.append(v_all[j])

            m = max(scores)
            exps = [math.exp(max(-50.0, min(50.0, s - m))) for s in scores]
            denom = max(sum(exps), 1e-12)
            context = sum(w * v for w, v in zip(exps, values)) / denom
            return round(gain * context)

        # No learned head has been committed. Leave the attention contribution at
        # zero so ordinary discrete / GA brains retain the original travel rule.
        return 0

    def compute_input(self, input):
        directions, positions, travel = [], [], []
        input_arr = TokenizeInput(input, self.alphabet)
        for token in input_arr:
            if token in self.alphabet:
                index = self.alphabet.index(token)
                directions.append(self.alphabet_directions[index])
                positions.append(self.alphabet_start_positions[index])
        for i in range(len(directions)):
            travel.append(self.travel(copy.copy(directions), copy.copy(positions), i))
        return self.ricochet(travel, 10, self.ball_energy)

    def ricochet(self, travel, cap, ball_energy):
        output = []
        dl = self.deflectors_list
        n = len(dl)

        for t in travel:
            # normalize index using modulo (handles large or negative values)
            new_idx = t % n
            cap_idx = 0
            be = ball_energy

            while be > 0 and cap_idx < cap:
                dest, val = dl[new_idx]
                distance = abs(dest - new_idx)

                if not isinstance(val, int):
                    output.append(val)
                    be += 1
                    be -= distance
                    new_idx = dest
                    cap_idx += 1
                    continue

                be += val
                be -= distance
                new_idx = dest
                cap_idx += 1

        return output

    def compute_deflectors(self, input):
        """
        Processes input exactly like compute_input but returns an array
        of all deflectors hit during the ball's travel.
        """
        directions = []
        positions = []
        travel = []
        input_arr = TokenizeInput(input, self.alphabet)

        for i in input_arr:
            if i in self.alphabet:
                index = self.alphabet.index(i)
                directions.append(self.alphabet_directions[index])
                positions.append(self.alphabet_start_positions[index])

        for i in range(len(directions)):
            travel.append(self.travel(copy.copy(directions), copy.copy(positions), i))

        # Passes the standard cap of 10 and the current ball_energy
        return self.ricochet_deflectors(travel, 10, self.ball_energy)

    def ricochet_deflectors(self, travel, cap, ball_energy):
        """
        Modified ricochet logic that records and returns the deflectors
        hit instead of the output values.
        """
        deflectors_hit = []
        dl = self.deflectors_list
        n = len(dl)

        for t in travel:
            # normalize index using modulo (handles large or negative values)
            new_idx = t % n
            cap_idx = 0
            be = ball_energy

            while be > 0 and cap_idx < cap:
                dest, val = dl[new_idx]

                # Append the deflector's details to our tracking array
                deflectors_hit.append(new_idx)

                distance = abs(dest - new_idx)

                if not isinstance(val, int):
                    be += 1
                    be -= distance
                    new_idx = dest
                    cap_idx += 1
                    continue

                be += val
                be -= distance
                new_idx = dest
                cap_idx += 1

        return deflectors_hit

    def travel(self, directions, positions, index):
        """Return a token's starting deflector using the discrete attention head."""
        position = positions[index]
        if not directions:
            return position
        if self.attention_enabled:
            return position + self._single_head_attention_discrete(directions, positions, index)

        # Original fallback: fixed leave-one-out positional weighting.
        total = 0
        weight_sum = 0
        for j, d in enumerate(directions):
            if j == index:
                continue
            weight = j + 1
            total += d * weight
            weight_sum += weight
        if weight_sum > 0:
            total = round(total / weight_sum)
        return position + total

    def display_deflectors(self):
        for i, deflect in enumerate(self.deflectors_list):
            print(f"Deflector {i}: {deflect}")

    def clone(self):
      # Create a lightweight dummy instance
      new_brain = Ball_Trap_Brain(1, self.ball_energy // 2, self.alphabet, self.output,
                                   self.holes_multiplier, self.direction_range)

      # Fast shallow copy of the 10,000 elements
      new_brain.attention_enabled = self.attention_enabled
      new_brain.attention_gain = self.attention_gain
      if hasattr(self, '_discrete_q_proj'):
          new_brain._discrete_q_proj = [row[:] for row in self._discrete_q_proj]
          new_brain._discrete_k_proj = [row[:] for row in self._discrete_k_proj]
          new_brain._discrete_v_proj = [row[:] for row in self._discrete_v_proj]
          new_brain._discrete_attn_gain = self._discrete_attn_gain
      new_brain.deflectors_list = [[d, v] for d, v in self.deflectors_list]
      new_brain.alphabet_directions = self.alphabet_directions[:]
      new_brain.alphabet_start_positions = self.alphabet_start_positions[:]

      return new_brain

    def shift_alphabet(self):
        random.shuffle(self.alphabet)
        for i in range(len(self.deflectors_list)):
          if self.deflectors_list[i][1] in self.output:
            idx = self.output.index(self.deflectors_list[i][1])
            self.deflectors_list[i][1] = self.alphabet[idx]

    def Mutate(self, individual_probability=0.1, probability=0.05):
        if random.random() < individual_probability:
            for i in range(len(self.deflectors_list)):
                  if random.random() < probability:
                      self.deflectors_list[i][0] = random.randint(0, len(self.deflectors_list)-1)

            for i in range(len(self.deflectors_list)):
                 if random.random() < probability:
                     self.deflectors_list[random.randint(0, len(self.deflectors_list)-1)][1] = random.choice(self.output)

            for i in range(len(self.alphabet_directions)):
                 if random.random() < probability:
                     self.alphabet_directions[i] = random.randint(-self.direction_range, self.direction_range)

            for i in range(len(self.alphabet_start_positions)):
                  if random.random() < probability:
                      self.alphabet_start_positions[i] = random.randint(0, len(self.deflectors_list)-1)

            if random.random() < probability:
                  self.set_holes(self.output, len(self.deflectors_list), self.holes_multiplier)

        return self

    @staticmethod
    def _resolve_batch_size(batch_size, frac):
        """Turn a `batch_size` spec into a concrete int for this unit (epoch/step).

        `batch_size` may be:
          - an int -> returned as-is (the original, constant behavior).
          - a (start, end) tuple/list -> linearly interpolated at `frac` (0..1).
          - a callable f(frac) -> int -> called directly.
        The result is always rounded to the nearest int and floored at 1, so a
        schedule can never collapse to an empty (or negative) batch.
        """
        if callable(batch_size):
            bs = batch_size(frac)
        elif isinstance(batch_size, (tuple, list)):
            bs_start, bs_end = batch_size
            bs = bs_start + (bs_end - bs_start) * frac
        else:
            bs = batch_size
        return max(1, int(round(bs)))


    def train_continuous(self, test_cases, epochs=300, steps=None, lr_start=0.05, lr_end=None, batch_size=16,
                          temp_start=2.5, temp_end=0.5, sharpness=0.75,
                          index_temp_start=None, index_temp_end=0.05, index_temp_power=2.0,
                          grad_clip_norm=1.0, restore_best=True,
                          repeat_penalty_weight=0.3, blank_loss_weight=1.0, device=None,
                          warm_start=True, commit=True, verbose=True, save_path=None):
        """Train the continuous relaxation with a genuinely batched forward pass.

        Each training batch is padded to a common sequence length `Lmax` and passed
        through `_c_simulate_batch`, which keeps the simulation state as
        `(B, Lmax, D)`. The `cap=10` rollout therefore performs the same trajectory
        math for all examples in the batch together instead of calling the simulator
        once per example. Padding is masked out of the simulation, loss, and metrics.

        The optimization objective, temperature/index-temperature schedules, gradient
        clipping, best-checkpoint restore, continuous parameterization, and optional
        discretization are otherwise unchanged.
        """
        import torch
        import math

        has_continuous = hasattr(self, "_c_meta")


        self._init_continuous_params(warm_start=warm_start, sharpness=sharpness, device=device)


        c = self._c_meta
        output_index = c['output_index']

        params = [self._c_dest_logits, self._c_hole_logit, self._c_symbol_logits,
                  self._c_bounce_val, self._c_direction_logits, self._c_start_logits,
                  self._c_q_proj, self._c_k_proj, self._c_v_proj, self._c_attn_gain]

        if hasattr(self, "_c_optimizer"):
            optimizer = self._c_optimizer
        else:
            optimizer = torch.optim.Adam(params, lr=lr_start)
            self._c_optimizer = optimizer

        if index_temp_start is None:
            index_temp_start = temp_start
        if lr_end is None:
            lr_end = lr_start / 20.0

        best_loss = float('inf')
        best_state = None
        history = []
        acc_history = []
        temp = temp_start
        index_temp = index_temp_start

        use_steps = steps is not None
        total_units = steps if use_steps else epochs
        unit_label = 'step' if use_steps else 'epoch'
        if total_units <= 0:
            return history, acc_history

        step_pool = []
        step_cursor = 0

        for unit in range(total_units):
            frac = unit / max(1, total_units - 1)
            temp = temp_start + (temp_end - temp_start) * frac
            index_temp = index_temp_start + (index_temp_end - index_temp_start) * (frac ** index_temp_power)
            lr_now = lr_end + 0.5 * (lr_start - lr_end) * (1 + math.cos(math.pi * frac))
            for group in optimizer.param_groups:
                group['lr'] = lr_now

            batch_size_now = self._resolve_batch_size(batch_size, frac)

            if use_steps:
                if step_cursor + batch_size_now > len(step_pool):
                    step_pool = list(test_cases)
                    random.shuffle(step_pool)
                    step_cursor = 0
                batch = step_pool[step_cursor:step_cursor + batch_size_now]
                step_cursor += batch_size_now
                batches = [batch]
            else:
                shuffled = list(test_cases)
                random.shuffle(shuffled)
                n_batches_now = max(1, (len(shuffled) + batch_size_now - 1) // batch_size_now)
                batches = [shuffled[b * batch_size_now:(b + 1) * batch_size_now]
                           for b in range(n_batches_now)]

            epoch_loss_sum = 0.0
            epoch_count = 0
            epoch_correct = 0
            epoch_fire_correct = 0
            epoch_fire_count = 0

            for batch in batches:
                if not batch:
                    continue

                # Tokenize/build targets in Python, but do NOT run the differentiable
                # simulator here. The whole batch is packed once and simulated once.
                packed_indices = []
                target_indices_batch = []
                valid_lengths = []
                filtered_batch = []

                for inp, target in batch:
                    tokens = TokenizeInput(inp, self.alphabet)
                    indices = [self.alphabet.index(t) for t in tokens if t in self.alphabet]
                    if not indices:
                        continue
                    packed_indices.append(indices)
                    target_indices_batch.append([output_index[ch] for ch in target if ch in output_index])
                    valid_lengths.append(len(indices))
                    filtered_batch.append((inp, target))

                if not packed_indices:
                    continue

                B = len(packed_indices)
                Lmax = max(valid_lengths)
                dev = self._c_dest_logits.device

                # One `(B, Lmax)` index tensor and one validity mask. Padding uses index 0
                # only as a harmless gather placeholder; `valid_mask` removes it from
                # the simulation and all reductions below.
                input_indices_t = torch.zeros((B, Lmax), dtype=torch.long, device=dev)
                valid_mask = torch.zeros((B, Lmax), dtype=torch.bool, device=dev)
                for b, indices in enumerate(packed_indices):
                    n = len(indices)
                    input_indices_t[b, :n] = torch.as_tensor(indices, dtype=torch.long, device=dev)
                    valid_mask[b, :n] = True

                target_class_t = torch.full((B, Lmax), c['K'], dtype=torch.long, device=dev)
                for b, (L, target_indices) in enumerate(zip(valid_lengths, target_indices_batch)):
                    n = min(L, len(target_indices))
                    if n:
                        out_offset = L - n
                        tgt_offset = len(target_indices) - n
                        target_class_t[b, out_offset:L] = torch.as_tensor(
                            target_indices[tgt_offset:tgt_offset + n], dtype=torch.long, device=dev
                        )

                optimizer.zero_grad()
                precomputed = self._c_precompute(temp)
                outs, never_fired, extra_fire = self._c_simulate_batch(
                    input_indices_t, valid_mask, temp, index_temp, precomputed=precomputed
                )

                is_blank = target_class_t == c['K']
                scored_mask = valid_mask
                fire_mask = scored_mask & ~is_blank
                blank_mask = scored_mask & is_blank

                safe_idx = target_class_t.clamp(max=c['K'] - 1)
                gathered = outs.gather(2, safe_idx.unsqueeze(-1)).squeeze(-1)
                target_prob = torch.where(is_blank, never_fired, gathered)
                loss_per_pos = -torch.log(target_prob + 1e-8)

                fire_loss_sum = loss_per_pos.masked_select(fire_mask).sum()
                fire_count_t = fire_mask.sum().to(loss_per_pos.dtype)
                blank_loss_sum = loss_per_pos.masked_select(blank_mask).sum()
                blank_count_t = blank_mask.sum().to(loss_per_pos.dtype)

                fire_loss_mean = fire_loss_sum / fire_count_t.clamp(min=1)
                blank_loss_mean = blank_loss_sum / blank_count_t.clamp(min=1)
                task_loss = fire_loss_mean + blank_loss_weight * blank_loss_mean

                combined_loss = task_loss
                if repeat_penalty_weight:
                    repeat_sum = extra_fire.masked_select(scored_mask).sum()
                    scored_count_t = scored_mask.sum().to(repeat_sum.dtype)
                    combined_loss = combined_loss + repeat_penalty_weight * (repeat_sum / scored_count_t.clamp(min=1))

                combined_loss.backward()
                if grad_clip_norm is not None:
                    torch.nn.utils.clip_grad_norm_(params, grad_clip_norm)
                optimizer.step()

                with torch.no_grad():
                    p_emit = outs.sum(dim=2)
                    pred_symbol = outs.argmax(dim=2)
                    pred_class = torch.where(
                        p_emit >= 0.5, pred_symbol, torch.full_like(pred_symbol, c['K'])
                    )
                    is_correct = (pred_class == target_class_t) & scored_mask

                    count = int(scored_mask.sum().item())
                    correct = int(is_correct.sum().item())
                    fire_correct = int((is_correct & fire_mask).sum().item())
                    fire_count = int(fire_mask.sum().item())

                epoch_loss_sum += float(task_loss.item()) * count
                epoch_count += count
                epoch_correct += correct
                epoch_fire_correct += fire_correct
                epoch_fire_count += fire_count

            if epoch_count == 0:
                continue

            avg_loss = epoch_loss_sum / epoch_count
            avg_acc = epoch_correct / epoch_count
            avg_fire_acc = (epoch_fire_correct / epoch_fire_count
                            if epoch_fire_count else float('nan'))
            history.append(avg_loss)
            acc_history.append(avg_acc)

            if restore_best and avg_loss < best_loss:
                best_loss = avg_loss
                best_state = [p.detach().clone() for p in params]

            if verbose and (unit % max(1, total_units // 20) == 0 or unit == total_units - 1):
                print(f"[continuous] {unit_label} {unit}: loss={avg_loss:.4f} token_acc={avg_acc:.3f} "
                      f"fire_acc={avg_fire_acc:.3f} temp={temp:.2f} index_temp={index_temp:.3f} "
                      f"lr={lr_now:.5f} batch_size={batch_size_now} batches={len(batches)}")

        self._c_meta['last_temp'] = temp
        self._c_meta['last_index_temp'] = index_temp

        if restore_best and best_state is not None:
            with torch.no_grad():
                for p, saved in zip(params, best_state):
                    p.copy_(saved)
            if verbose:
                print(f"[continuous] restored best checkpoint: loss={best_loss:.4f} "
                      f"(final epoch's loss was {history[-1]:.4f})")

        if commit:
            self._discretize_continuous_into(self)

        if save_path is not None:
            self.save_checkpoint(save_path)
            if verbose:
                print(f"[continuous] saved checkpoint to {save_path!r}")

        return history, acc_history

    def _init_continuous_params(self, warm_start=True, sharpness=0.75, device=None):
        """Initialize continuous relaxation plus one learned Q/K/V attention head."""
        import torch
        if device is None:
            device = 'cuda' if torch.cuda.is_available() else 'cpu'
        D, A, K = len(self.deflectors_list), len(self.alphabet), len(self.output)
        output_index = {sym: k for k, sym in enumerate(self.output)}
        positions = torch.arange(D, dtype=torch.float32, device=device)
        abs_diff = (positions[:, None] - positions[None, :]).abs()
        V = self.direction_range
        n_dirs = 2 * V + 1
        direction_values = torch.arange(-V, V + 1, dtype=torch.float32, device=device)

        CONF = 4.0
        dest_logits = torch.zeros(D, D, device=device)
        hole_logit = torch.zeros(D, device=device)
        symbol_logits = torch.zeros(D, K, device=device)
        bounce_val = torch.zeros(D, device=device)
        direction_logits = torch.zeros(A, n_dirs, device=device)
        start_logits = torch.zeros(A, D, device=device)

        # Token feature vector = normalized direction, normalized start, bias.
        ATT_DIM = 8
        q_proj = torch.zeros(3, ATT_DIM, device=device)
        k_proj = torch.zeros(3, ATT_DIM, device=device)
        v_proj = torch.zeros(3, 1, device=device)
        attn_gain = torch.tensor(1.0, dtype=torch.float32, device=device)

        if warm_start:
            for i, (dest, val) in enumerate(self.deflectors_list):
                dest_logits[i, dest] = CONF
                if isinstance(val, int):
                    hole_logit[i] = -CONF
                    bounce_val[i] = float(val)
                else:
                    hole_logit[i] = CONF
                    symbol_logits[i, output_index[val]] = CONF
            for i, d in enumerate(self.alphabet_directions):
                direction_logits[i, max(0, min(n_dirs - 1, int(d) + V))] = CONF
            for i, p in enumerate(self.alphabet_start_positions):
                start_logits[i, p] = CONF
            # Warm-start attention from a previously committed/trained discrete head
            # when available; otherwise begin with a tiny attention contribution.
            if hasattr(self, '_discrete_q_proj'):
                q_proj.copy_(torch.tensor(self._discrete_q_proj, dtype=q_proj.dtype, device=device))
                k_proj.copy_(torch.tensor(self._discrete_k_proj, dtype=k_proj.dtype, device=device))
                v_proj.copy_(torch.tensor(self._discrete_v_proj, dtype=v_proj.dtype, device=device))
                attn_gain.fill_(float(self._discrete_attn_gain))
            else:
                q_proj[0, 0] = 0.10
                k_proj[0, 0] = 0.10
                v_proj[0, 0] = 0.25
        else:
            for t in (dest_logits, hole_logit, symbol_logits, direction_logits,
                      start_logits, q_proj, k_proj, v_proj):
                t.normal_(0, 0.1)
            bounce_val.normal_(0, 1.0)
            attn_gain.fill_(0.25)

        for t in (dest_logits, hole_logit, symbol_logits, bounce_val,
                  direction_logits, start_logits, q_proj, k_proj, v_proj, attn_gain):
            t.requires_grad_(True)

        self._c_dest_logits = dest_logits
        self._c_hole_logit = hole_logit
        self._c_symbol_logits = symbol_logits
        self._c_bounce_val = bounce_val
        self._c_direction_logits = direction_logits
        self._c_start_logits = start_logits
        self._c_q_proj = q_proj
        self._c_k_proj = k_proj
        self._c_v_proj = v_proj
        self._c_attn_gain = attn_gain
        self._c_meta = dict(
            D=D, A=A, K=K, cap=10, ball_energy=float(self.ball_energy),
            output_index=output_index, positions=positions, abs_diff=abs_diff,
            direction_values=direction_values, attention_dim=ATT_DIM,
            last_temp=None, last_index_temp=None, sharpness=sharpness
        )

    def _c_attention(self, dirs, starts, valid):
        """Single-head scaled-dot-product attention across a token sequence."""
        import torch
        import torch.nn.functional as F
        c = self._c_meta
        ds = max(1.0, float(self.direction_range))
        ps = max(1.0, float(c['D'] - 1))
        features = torch.stack((dirs / ds, starts / ps, torch.ones_like(dirs)), dim=-1)
        Q = torch.matmul(features, self._c_q_proj)
        K = torch.matmul(features, self._c_k_proj)
        values = torch.matmul(features, self._c_v_proj).squeeze(-1)
        scores = torch.matmul(Q, K.transpose(1, 2)) / math.sqrt(float(c['attention_dim']))

        B, L = dirs.shape
        self_mask = torch.eye(L, dtype=torch.bool, device=dirs.device).unsqueeze(0)
        key_mask = (~valid.bool()).unsqueeze(1)
        scores = scores.masked_fill(self_mask | key_mask, -1e9)

        attn = F.softmax(scores, dim=-1)
        has_context = (valid.sum(dim=1, keepdim=True) > 1).unsqueeze(-1)
        attn = attn.masked_fill(~has_context, 0.0)
        return torch.matmul(attn, values.unsqueeze(-1)).squeeze(-1) * valid

    def _c_soft_index_dist(self, travel_values, temp):
        """Batched: travel_values is a 1D tensor of shape (L,) (one travel value per
        input position); returns a (L, D) tensor, one starting-deflector
        distribution per row."""
        import torch
        import torch.nn.functional as F
        c = self._c_meta
        diff = c['positions'].unsqueeze(0) - travel_values.unsqueeze(1)  # (L, D)
        diff = torch.remainder(diff + c['D'] / 2, c['D']) - c['D'] / 2
        logits = -(diff ** 2) / (c['sharpness'] * temp)
        return F.softmax(logits, dim=1)

    def _c_precompute(self, temp):
        """Compute the tensors that _c_simulate needs but that only depend on the
        current parameter values and `temp` -- NOT on which example is being
        simulated: dest_probs, hole_p, symbol_probs, weighted_abs, and the
        expected per-token direction/start-position. `temp` is constant across an
        entire training epoch (it's only updated once per epoch, not per batch or
        per example), so previously calling _c_simulate separately for every
        example recomputed this same O(D^2) softmax (among others) from scratch
        once per example, all producing an identical result -- pure wasted work
        that scaled with batch_size * n_batches for no benefit. Call this once per
        epoch (or whenever `temp` changes) and pass the result into every
        _c_simulate call via its `precomputed` argument instead."""
        import torch
        import torch.nn.functional as F

        c = self._c_meta
        dest_logits, hole_logit = self._c_dest_logits, self._c_hole_logit
        symbol_logits, bounce_val = self._c_symbol_logits, self._c_bounce_val
        direction_logits, start_logits = self._c_direction_logits, self._c_start_logits

        direction_probs_all = F.softmax(direction_logits / temp, dim=1)             # (A, n_dirs)
        expected_direction_all = (direction_probs_all * c['direction_values']).sum(dim=1)  # (A,)

        start_probs_all = F.softmax(start_logits / temp, dim=1)    # (A, D)
        expected_start_all = (start_probs_all * c['positions']).sum(dim=1)  # (A,)

        dest_probs = F.softmax(dest_logits / temp, dim=1)          # (D, D)
        hole_p = torch.sigmoid(hole_logit)                         # (D,)
        symbol_probs = F.softmax(symbol_logits / temp, dim=1)      # (D, K)
        weighted_abs = (dest_probs * c['abs_diff']).sum(dim=1)     # (D,) expected hop distance

        return dict(expected_direction_all=expected_direction_all, expected_start_all=expected_start_all,
                    dest_probs=dest_probs, hole_p=hole_p, symbol_probs=symbol_probs,
                    weighted_abs=weighted_abs, bounce_val=bounce_val)


    def _c_simulate_batch(self, input_indices, valid_mask, temp, index_temp=None, precomputed=None):
        """Differentiable batched simulation.

        Args:
            input_indices: Long tensor `(B, Lmax)` containing alphabet indices.
            valid_mask: Bool tensor `(B, Lmax)` identifying real (unpadded) positions.
            temp: temperature for learnable-logit softmaxes.
            index_temp: temperature for the non-learnable travel-to-index kernel.

        Returns:
            outs: `(B, Lmax, K)` distribution for the first hole reached by each
                trajectory, with zeros on padded positions.
            never_fired: `(B, Lmax)` probability of no emitted symbol within `cap`.
            extra_fire: `(B, Lmax)` expected additional hole hits after the first.

        The rollout is performed for all B examples and all Lmax trajectories in
        parallel. Only the padding mask prevents padded rows from contributing; there
        is no Python-level simulation loop over examples.
        """
        import torch

        if index_temp is None:
            index_temp = temp
        c = self._c_meta
        if precomputed is None:
            precomputed = self._c_precompute(temp)

        expected_direction_all = precomputed['expected_direction_all']
        expected_start_all = precomputed['expected_start_all']
        dest_probs = precomputed['dest_probs']
        hole_p = precomputed['hole_p']
        symbol_probs = precomputed['symbol_probs']
        weighted_abs = precomputed['weighted_abs']
        bounce_val = precomputed['bounce_val']

        valid = valid_mask.to(dtype=dest_probs.dtype)
        B, L = input_indices.shape
        K = c['K']

        # Per-position expected direction/start values: `(B, L)`. Padded entries are
        # gathered from index 0 but immediately zeroed by `valid`.
        dirs = expected_direction_all[input_indices] * valid
        starts = expected_start_all[input_indices] * valid

        # Learned single-head attention replaces the fixed positional weighting.
        # Each token attends to other tokens only (self-attention is masked).
        attn_context = self._c_attention(dirs, starts, valid)
        attn_context = torch.tanh(self._c_attn_gain) * attn_context
        travel_all = starts + attn_context
        travel_all = torch.where(valid > 0, travel_all, torch.zeros_like(travel_all))
        state = self._c_soft_index_dist(travel_all.reshape(-1), index_temp).reshape(B, L, c['D'])
        state = state * valid.unsqueeze(-1)

        be = torch.full((B, L), c['ball_energy'], dtype=state.dtype, device=state.device)
        alive = valid.clone()
        not_hole_yet = valid.clone()
        out_dist = torch.zeros(B, L, K, dtype=state.dtype, device=state.device)
        extra = torch.zeros(B, L, dtype=state.dtype, device=state.device)

        for _ in range(c['cap']):
            p_hole = (state * hole_p.view(1, 1, -1)).sum(dim=2)
            p_bounce_energy = (state * (1 - hole_p).view(1, 1, -1)
                               * bounce_val.view(1, 1, -1)).sum(dim=2)
            exp_dist = (state * weighted_abs.view(1, 1, -1)).sum(dim=2)

            already_fired = alive * (1.0 - not_hole_yet)
            extra = extra + already_fired * p_hole

            p_first_hole = alive * not_hole_yet * p_hole
            sym_dist = torch.matmul(state * hole_p.view(1, 1, -1), symbol_probs)
            sym_dist = sym_dist / (p_hole.unsqueeze(-1) + 1e-8)
            out_dist = out_dist + p_first_hole.unsqueeze(-1) * sym_dist

            energy_delta = p_hole + p_bounce_energy - exp_dist
            be = be + energy_delta
            alive = alive * torch.sigmoid(be * 2.0)
            not_hole_yet = not_hole_yet * (1 - p_hole)

            # `(B, L, D) @ (D, D) -> (B, L, D)`: the expensive routing operation
            # is now one batched matrix multiply for the whole batch.
            state = torch.matmul(state, dest_probs)
            state = state * valid.unsqueeze(-1)

        never_fired = 1.0 - out_dist.sum(dim=2)
        return out_dist, never_fired, extra


    def _c_simulate(self, input_indices, temp, index_temp=None, precomputed=None):
        """Compatibility wrapper for one example.

        The actual implementation lives in `_c_simulate_batch`; this method simply
        supplies a batch dimension and unpacks the result so existing prediction code
        keeps its original API. Training never calls this per example anymore.
        """
        import torch
        L = len(input_indices)
        dev = self._c_dest_logits.device
        if L == 0:
            K = self._c_meta['K']
            empty_k = torch.empty((0, K), device=dev)
            empty_1 = torch.empty((0,), device=dev)
            return list(empty_k.unbind(0)), list(empty_1.unbind(0)), list(empty_1.unbind(0))
        indices_t = torch.as_tensor(input_indices, dtype=torch.long, device=dev).unsqueeze(0)
        valid_mask = torch.ones((1, L), dtype=torch.bool, device=dev)
        outs, never_fired, extra_fire = self._c_simulate_batch(
            indices_t, valid_mask, temp, index_temp=index_temp, precomputed=precomputed
        )
        return list(outs[0].unbind(0)), list(never_fired[0].unbind(0)), list(extra_fire[0].unbind(0))

    def _discretize_continuous_into(self, target_brain):
        """Argmax/sign/round this instance's continuous parameters (self._c_*) and write
        them into target_brain's discrete attributes (deflectors_list, alphabet_directions,
        alphabet_start_positions)."""
        import torch
        c = self._c_meta
        target_brain.direction_range = self.direction_range
        with torch.no_grad():
            for i in range(c['D']):
                dest = int(torch.argmax(self._c_dest_logits[i]).item())
                if torch.sigmoid(self._c_hole_logit[i]).item() > 0.5:
                    val = self.output[int(torch.argmax(self._c_symbol_logits[i]).item())]
                else:
                    val = int(round(self._c_bounce_val[i].item()))
                target_brain.deflectors_list[i][0] = dest
                target_brain.deflectors_list[i][1] = val
            for i in range(c['A']):
                dir_idx = int(torch.argmax(self._c_direction_logits[i]).item())
                target_brain.alphabet_directions[i] = int(c['direction_values'][dir_idx].item())
                target_brain.alphabet_start_positions[i] = int(torch.argmax(self._c_start_logits[i]).item())

            # IMPORTANT: transfer the trained attention head too. The old commit
            # operation discarded Q/K/V entirely, so the discrete brain then used a
            # different attention function from the continuous brain.
            target_brain.attention_enabled = True
            target_brain._discrete_q_proj = self._c_q_proj.detach().cpu().tolist()
            target_brain._discrete_k_proj = self._c_k_proj.detach().cpu().tolist()
            target_brain._discrete_v_proj = self._c_v_proj.detach().cpu().tolist()
            target_brain._discrete_attn_gain = float(torch.tanh(self._c_attn_gain).item())

    def commit_continuous(self):
        """Discretize this instance's already-trained continuous parameters (from a prior
        train_continuous(..., commit=False) call) into its discrete deflectors_list /
        alphabet_directions / alphabet_start_positions, without training any further."""
        if not hasattr(self, '_c_meta'):
            raise RuntimeError("No continuous parameters to commit -- call train_continuous(...) first.")
        self._discretize_continuous_into(self)

    def save_checkpoint(self, path):
        """Save this brain to `path` (via pickle) so it can be reloaded later and
        trained *again* -- across cells, kernel restarts, or completely separate
        sessions -- instead of losing all progress once a `train_continuous` call
        returns. This is the piece that makes "keep training until the loss hits
        (near) zero" practical without one single, uninterruptible run.

        Two layers of state are saved:
          - The discrete state every non-training method (`compute_input`,
            `clone`, `Mutate`, ...) relies on: `deflectors_list`,
            `alphabet_directions`, `alphabet_start_positions`, plus the
            constructor-level settings (`alphabet`, `output`,
            `holes_multiplier`, `ball_energy`, `direction_range`) needed to
            reconstruct a matching instance.
          - If this instance has already run `train_continuous` at least once
            (i.e. it has continuous parameters), their *trained* values --
            `_c_dest_logits`, `_c_hole_logit`, `_c_symbol_logits`,
            `_c_bounce_val`, `_c_direction_logits`, `_c_start_logits` -- plus
            `last_temp`/`last_index_temp`/`sharpness` from `_c_meta`. This is
            what lets a resumed run pick back up from where the logits
            actually were, rather than re-warm-starting them from the
            (coarser) discrete state at a flat, fixed confidence every time.
            If `train_continuous` was never called (or only the genetic
            algorithm was used), only the discrete layer is saved -- and a
            resumed run just warm-starts continuous training from that
            discrete state as normal, same as calling `train_continuous`
            fresh on any other already-evolved/discrete brain.

        Tensors are detached and moved to CPU before saving, so a checkpoint
        saved on a CUDA machine can still be loaded on a CPU-only one (and
        vice versa via `load_checkpoint`'s `device` argument).
        """
        import pickle
        import os

        state = {
            'version': 1,
            'alphabet': self.alphabet,
            'output': self.output,
            'holes_multiplier': self.holes_multiplier,
            'ball_energy': self.ball_energy,
            'direction_range': self.direction_range,
            'attention_enabled': self.attention_enabled,
            'attention_gain': self.attention_gain,
            'discrete_attention': {
                'q_proj': self._discrete_q_proj,
                'k_proj': self._discrete_k_proj,
                'v_proj': self._discrete_v_proj,
                'attn_gain': self._discrete_attn_gain,
            } if hasattr(self, '_discrete_q_proj') else None,
            'deflectors_list': self.deflectors_list,
            'alphabet_directions': self.alphabet_directions,
            'alphabet_start_positions': self.alphabet_start_positions,
        }

        if hasattr(self, '_c_meta'):
            state['continuous'] = {
                'dest_logits': self._c_dest_logits.detach().cpu(),
                'hole_logit': self._c_hole_logit.detach().cpu(),
                'symbol_logits': self._c_symbol_logits.detach().cpu(),
                'bounce_val': self._c_bounce_val.detach().cpu(),
                'direction_logits': self._c_direction_logits.detach().cpu(),
                'start_logits': self._c_start_logits.detach().cpu(),
                'q_proj': self._c_q_proj.detach().cpu(),
                'k_proj': self._c_k_proj.detach().cpu(),
                'v_proj': self._c_v_proj.detach().cpu(),
                'attn_gain': self._c_attn_gain.detach().cpu(),
                'last_temp': self._c_meta.get('last_temp'),
                'last_index_temp': self._c_meta.get('last_index_temp'),
                'sharpness': self._c_meta.get('sharpness'),
            }

        if hasattr(self, "_c_optimizer"):
            state["optimizer"] = self._c_optimizer.state_dict()

        dirname = os.path.dirname(path)
        if dirname:
            os.makedirs(dirname, exist_ok=True)
        with open(path, 'wb') as f:
            pickle.dump(state, f)

    @classmethod
    def load_checkpoint(cls, path, device=None):
        """Reconstruct a `Ball_Trap_Brain` from a file written by `save_checkpoint`
        (or by `train_continuous(..., save_path=...)`). The discrete state is
        always restored; if the checkpoint also has trained continuous
        parameters, they're restored too (via a fresh `_init_continuous_params`
        call to rebuild the non-learnable meta tensors, then overwriting its
        learnable parameters with the saved, trained values) so a follow-up
        `train_continuous(...)` call resumes from them directly instead of
        warm-starting from scratch.

        `device` defaults to CUDA if available, else CPU, same as
        `_init_continuous_params` -- pass `device='cpu'` explicitly to force
        CPU regardless of what's available, e.g. to load a checkpoint on a
        machine without a GPU.
        """
        import pickle

        with open(path, 'rb') as f:
            state = pickle.load(f)

        brain = cls.__new__(cls)  # bypass __init__ -- we're restoring, not randomly initializing
        brain.alphabet = state['alphabet']
        brain.output = state['output']
        brain.holes_multiplier = state['holes_multiplier']
        brain.ball_energy = state['ball_energy']
        brain.direction_range = state['direction_range']
        brain.attention_enabled = state.get('attention_enabled', True)
        brain.attention_gain = state.get('attention_gain', 1.0)
        brain.deflectors_list = state['deflectors_list']
        brain.alphabet_directions = state['alphabet_directions']
        brain.alphabet_start_positions = state['alphabet_start_positions']
        discrete_attention = state.get('discrete_attention')
        if discrete_attention:
            brain._discrete_q_proj = discrete_attention['q_proj']
            brain._discrete_k_proj = discrete_attention['k_proj']
            brain._discrete_v_proj = discrete_attention['v_proj']
            brain._discrete_attn_gain = float(discrete_attention['attn_gain'])

        cont = state.get('continuous')
        if cont is not None:
            import torch
            if device is None:
                device = 'cuda' if torch.cuda.is_available() else 'cpu'
            # Rebuild the meta/derived tensors (positions, abs_diff, output_index, ...)
            # fresh -- cheap, and fully determined by the discrete state already
            # restored above -- then overwrite the actual learnable parameters
            # with the saved, trained values instead of the freshly
            # warm-started ones this call would otherwise produce.
            brain._init_continuous_params(warm_start=True, sharpness=cont.get('sharpness', 0.75), device=device)
            with torch.no_grad():
                brain._c_dest_logits.copy_(cont['dest_logits'].to(device))
                brain._c_hole_logit.copy_(cont['hole_logit'].to(device))
                brain._c_symbol_logits.copy_(cont['symbol_logits'].to(device))
                brain._c_bounce_val.copy_(cont['bounce_val'].to(device))
                brain._c_direction_logits.copy_(cont['direction_logits'].to(device))
                brain._c_start_logits.copy_(cont['start_logits'].to(device))
                if 'q_proj' in cont:
                    brain._c_q_proj.copy_(cont['q_proj'].to(device))
                    brain._c_k_proj.copy_(cont['k_proj'].to(device))
                    brain._c_v_proj.copy_(cont['v_proj'].to(device))
                    brain._c_attn_gain.copy_(cont['attn_gain'].to(device))
            brain._c_meta['last_temp'] = cont.get('last_temp')
            brain._c_meta['last_index_temp'] = cont.get('last_index_temp')
            if not discrete_attention:
                brain._discrete_q_proj = brain._c_q_proj.detach().cpu().tolist()
                brain._discrete_k_proj = brain._c_k_proj.detach().cpu().tolist()
                brain._discrete_v_proj = brain._c_v_proj.detach().cpu().tolist()
                brain._discrete_attn_gain = float(torch.tanh(brain._c_attn_gain).item())

        return brain

    def predict_continuous(self, input_str, temp=None, index_temp=None, return_probs=False):
        """
        Answer `input_str` using the *continuous* (un-discretized) brain directly --
        no argmax/rounding of parameters, no dependency on deflectors_list/
        alphabet_directions/alphabet_start_positions at all. Requires a prior
        train_continuous(...) call (with or without commit=True) on this instance.

        Unlike training (which only scores a target-aligned tail of each trajectory),
        this decodes *every* input character's trajectory: if its "never fire" probability
        is below 0.5 it's treated as emitting, and the most likely symbol from its output
        distribution is appended, in the same left-to-right order compute_input uses.

        `temp` defaults to the (logit-softmax) temperature training ended at, and
        `index_temp` defaults to the separately-annealed index-kernel temperature
        training ended at (see train_continuous's docstring for why these are two
        different schedules) -- colder is closer to the discretized/deterministic
        behavior; pass higher values to see softer, more exploratory behavior. Set
        `return_probs=True` to also get back, for every input character,
        `(probability it emits at all, {symbol: probability})`.
        """
        import torch

        if not hasattr(self, '_c_meta'):
            raise RuntimeError(
                "No continuous parameters on this brain yet -- call train_continuous(...) "
                "first (pass commit=False if you don't want it to also overwrite the "
                "discrete deflectors_list/alphabet_directions/alphabet_start_positions)."
            )
        c = self._c_meta
        if temp is None:
            temp = c['last_temp'] if c['last_temp'] is not None else 0.5
        if index_temp is None:
            index_temp = c['last_index_temp'] if c['last_index_temp'] is not None else 0.05

        tokens = TokenizeInput(input_str, self.alphabet)
        indices = [self.alphabet.index(t) for t in tokens if t in self.alphabet]
        if not indices:
            return ("", []) if return_probs else ""

        with torch.no_grad():
            outs, never_fired, _extra_fire = self._c_simulate(indices, temp, index_temp)

        decoded = []
        probs = []
        for out_dist, nf in zip(outs, never_fired):
            p_emit = 1.0 - nf.item()
            sym_probs = {sym: p for sym, p in zip(self.output, out_dist.tolist())}
            probs.append((p_emit, sym_probs))
            if p_emit >= 0.5:
                decoded.append(self.output[int(torch.argmax(out_dist).item())])

        result = "".join(decoded)
        return (result, probs) if return_probs else result

    def evaluate_continuous(self, test_cases, temp=None, index_temp=None, ratio=False):
        """Like evaluate_performance, but scores predict_continuous (the un-discretized
        brain) against test_cases instead of compute_input (the discretized/discrete
        brain). Returns the average score (0/1 exact-match per case, or a 0-1 similarity
        ratio if ratio=True)."""
        total = 0.0
        for inp, answer in test_cases:
            pred = self.predict_continuous(inp, temp=temp, index_temp=index_temp)
            if ratio:
                total += difflib.SequenceMatcher(None, pred, answer).ratio()
            else:
                total += 1 if pred == answer else 0
        return total / len(test_cases)


## Checkpointing & resumable training

`Ball_Trap_Brain.save_checkpoint` / `Ball_Trap_Brain.load_checkpoint` persist a brain (discrete state, plus trained continuous parameters if any) to disk so training can be picked back up later instead of being confined to one long `train_continuous` call. `train_continuous(..., save_path=...)` saves automatically when it finishes.

`train_to_convergence` below is a small loop built on top of that: it keeps calling `train_continuous` on the same (checkpointed) brain, round after round, until the loss reaches `target_loss` (as close to zero as you want) or `max_rounds` is hit -- so a single call can drive the loss all the way down without you having to manually save/reload between rounds.

In [ ]:
def train_to_convergence(brain, test_cases, checkpoint_path, target_loss=0.01,
                          max_rounds=20, round_kwargs=None, verbose=True):
    """Repeatedly call `train_continuous` on `brain`, checkpointing to
    `checkpoint_path` after every round (via `train_continuous`'s own
    `save_path=`), until either that round's epoch/step-average loss drops
    to `target_loss` or `max_rounds` rounds have run.

    Because `save_checkpoint`/`load_checkpoint` preserve the *trained*
    continuous relaxation parameters (not just the discretized brain),
    each round picks back up close to where the last one left off instead
    of re-warm-starting from scratch -- so this is meant to let the loss
    keep dropping across many rounds (each with its own fresh temp/lr
    anneal via `round_kwargs`) rather than one single long run, and to
    survive being stopped and restarted (e.g. across sessions) since every
    round's progress is saved to `checkpoint_path` before the next begins.

    `round_kwargs` is passed through to every `train_continuous` call
    (e.g. `dict(epochs=100, batch_size=(8, 64), lr_start=0.2, temp_start=1.5,
    temp_end=0.3)`) -- typically a shorter/gentler schedule per round than
    you'd use for a single one-shot run, since there are `max_rounds` of
    them. `save_path` is always controlled internally (set to
    `checkpoint_path`), so don't pass it inside `round_kwargs`.

    Returns `(brain, full_loss_history, full_acc_history)`, where the two
    histories are every round's per-epoch/per-step history concatenated
    end to end, in order, so you can see progress across the whole run.
    """
    round_kwargs = dict(round_kwargs or {})
    round_kwargs.pop('save_path', None)  # always controlled internally, see above

    full_loss_history = []
    full_acc_history = []

    for round_num in range(1, max_rounds + 1):
        if verbose:
            print(f"\n=== train_to_convergence: round {round_num}/{max_rounds} ===")
        loss_history, acc_history = brain.train_continuous(
            test_cases, save_path=checkpoint_path, verbose=verbose, **round_kwargs
        )
        full_loss_history.extend(loss_history)
        full_acc_history.extend(acc_history)

        final_loss = loss_history[-1] if loss_history else float('inf')
        if final_loss <= target_loss:
            if verbose:
                print(f"[train_to_convergence] reached target_loss={target_loss} "
                      f"(loss={final_loss:.4f}) after {round_num} round(s); stopping.")
            break
    else:
        if verbose:
            last = full_loss_history[-1] if full_loss_history else float('nan')
            print(f"[train_to_convergence] stopped after max_rounds={max_rounds} "
                  f"without reaching target_loss={target_loss} (final loss={last:.4f})")

    return brain, full_loss_history, full_acc_history


## Test case generation

In [ ]:

import random
import string
x = """Continuous relaxation training.

        Every discrete choice the brain makes is replaced with a differentiable
        stand-in for the duration of training:
          - each deflector's destination index      -> a softmax over all deflectors
          - each deflector's hole/bounce identity    -> a sigmoid (hole probability)
          - each deflector's output symbol (if hole) -> a softmax over self.output
          - each deflector's bounce value (if not)   -> a free real-valued scalar
          - each alphabet direction (integer in [-direction_range, direction_range]) -> a
            softmax over that candidate range (same pattern as the start position below)
          - each alphabet start position              -> a softmax over all deflectors

        The ricochet loop is unrolled for `cap` steps and the hard indexing /
        branching / while-loop of the discrete simulator is replaced by
        probability-weighted ("expected value") versions of the same
        arithmetic, so the whole forward pass is differentiable and can be
        trained with gradient descent against the task loss instead of the
        genetic algorithm's mutate-and-select loop. A temperature is annealed
        from `temp_start` down to `temp_end` over training so the softmax
        distributions sharpen towards one-hot (i.e. towards the original
        discrete model) as training progresses.

        **Epochs vs. steps.** Each epoch is a full pass over a shuffled copy of
        `test_cases`, split into batches of `batch_size`; the optimizer takes one
        gradient step per batch (so an epoch is several steps, not one). Every
        example is seen exactly once per epoch, rather than each iteration
        drawing an independent random sample of the data (which gives no
        per-epoch coverage guarantee and can revisit or skip examples). The
        temperature schedule and the returned histories are still indexed by
        epoch (one entry per epoch, averaged across that epoch's batches/steps).

        Pass `steps=N` to train for exactly N gradient steps instead of a
        number of full passes -- useful when you want a training budget that
        doesn't depend on `len(test_cases)`, or want to compare runs at equal
        step counts across different dataset sizes. When `steps` is given, it
        takes over as the loop's unit: temp/index_temp/lr are annealed by
        `step/steps` instead of `epoch/epochs`, and `loss_history`/
        `accuracy_history` get one entry per step instead of one per epoch.
        Coverage is still epoch-like under the hood -- each step pulls the next
        `batch_size` examples off a shuffled copy of `test_cases`, reshuffling
        and starting a fresh pass whenever it runs out -- so examples are still
        seen roughly equally often, rather than each step drawing an
        independent random sample (which would revisit some examples far more
        than others over a long run). Leave `steps=None` (the default) to keep
        the original epoch-based behavior exactly as-is.

        Because the discrete model can emit a variable number of output
        symbols per input, this relaxation makes one simplifying assumption:
        it treats the *first hole reached* by each character's trajectory as
        that character's contribution to the output, and aligns those
        per-character predictions against the target string position by
        position, right-aligned (the last trajectory position lines up with
        the last target character, and so on).

        Every trajectory position is trained, not just the aligned ones.
        `never_fired = 1 - out_dist.sum()` (the probability a trajectory ends
        the simulation without ever emitting anything) turns each position's
        prediction into a genuine (K+1)-way distribution: the K real output
        symbols, plus one "emit nothing" class, all summing to 1 by
        construction. Leading input positions that fall outside the aligned
        window (i.e. that have no corresponding target character at all) are
        given "emit nothing" as their target class in that same distribution.
        Earlier versions of this method left those leading positions with no
        training signal whatsoever, so they were free to fire and leak
        arbitrary symbols into the real, full output regardless of how low
        the loss on the aligned positions got; a later attempt fixed that
        with a second, separately-weighted penalty term, but that fought the
        main loss for control of shared deflectors and could destabilize
        training. Folding "should this position emit at all" into the *same*
        per-position cross-entropy as the main task avoids both problems --
        there's one loss, and every position (aligned or not) has a
        well-defined target under it.

        There's a related, narrower pathology this doesn't cover: once a
        trajectory reaches a hole, this objective stops caring what happens
        for the rest of that trajectory's `cap` steps. In the real, discrete
        brain, if that trajectory keeps landing on holes afterward (a direct
        self-loop, a 2-cycle between two holes, or any longer cycle -- all
        equally invisible to the "first hole" objective above), `ricochet`
        just keeps re-appending symbols for the rest of the `cap` steps (e.g.
        `"3333333333"`, `"2323232323"`). `repeat_penalty_weight` (default 0.3)
        penalizes the expected number of *additional* hole-hits after the
        first one for every trajectory in the batch (`extra_fire` from
        `_c_simulate`) -- computed from the same per-example forward pass
        already used for the main loss, so it directly targets whatever
        cycle each specific trajectory is actually at risk of, rather than
        only guarding against direct self-loops.

        **`index_temp_start`/`index_temp_end`.** The per-step routing softmaxes
        (dest, symbol, hole, start-position logits) can always be sharpened by
        training regardless of `temp`, since their logits are free parameters that
        gradient descent can grow arbitrarily to compensate. But the very first
        step of each trajectory -- converting the continuous `travel_i` scalar into
        a distribution over starting deflectors, in `_c_soft_index_dist` -- has NO
        learnable parameters; its width is fixed purely by temperature/sharpness.
        If that temperature never anneals close to zero, this kernel stays diffuse
        for the entire run, so a meaningful fraction of probability mass leaks onto
        neighboring (wrong) starting deflectors no matter how well every other
        parameter is trained -- putting a hard floor under the loss and capping
        accuracy below 100%, however long training runs. `index_temp` is therefore
        annealed on its own, separate schedule from `temp`, defaulting to a much
        lower end value (`index_temp_end=0.05`) so this kernel actually reaches
        near one-hot by the end of training. `index_temp_start` defaults to
        `temp_start` if left as `None`.

        **Why loss could spike back up late in training (and the fixes below).**
        Every softmax's gradient w.r.t. its logits scales roughly like `1/temp`, so
        as `temp`/`index_temp` cool, the same size of parameter update produces
        increasingly large swings in output probabilities -- and the (non-learnable)
        `_c_soft_index_dist` kernel simultaneously goes from forgiving (wide, so small
        errors in the continuous `travel_i` estimate get gently split across
        neighboring deflectors) to unforgiving (narrow, so the same small error can
        dump all its mass onto the wrong deflector). With a flat learning rate and no
        clipping, this means training becomes progressively *less* stable right when
        it's cooling into its most precision-demanding phase, and a single overshoot
        can flip a class assignment and spike the loss. Four things address this:

        - `lr_start`/`lr_end`: the learning rate now decays (cosine) alongside
          `temp`/`index_temp` instead of staying flat, so the effective step size in
          probability-space doesn't grow as the softmaxes sharpen. Defaults to
          `lr_end = lr_start / 20`.
        - `grad_clip_norm`: gradients are clipped to this global norm (via
          `torch.nn.utils.clip_grad_norm_`) before every `optimizer.step()`, capping
          how large a single overshoot can be regardless of temperature. Set to
          `None` to disable.
        - `index_temp_power`: the index-kernel cooling schedule is no longer linear
          in epoch -- it's `index_temp_start + (index_temp_end - index_temp_start) *
          frac ** index_temp_power`. With the default `power=2.0` this spends more
          epochs in the middle-to-low range before dropping to its sharpest, giving
          `travel_i` more time to converge before precision demands peak, rather than
          tightening the kernel at a constant rate all the way to the end.
        - `restore_best`: the parameter state with the lowest epoch-average
          `task_loss` seen during training is checkpointed (in memory) and restored
          onto the instance at the end, so a late-training spike can't leave you with
          a worse model than one that existed earlier in the run. Set to `False` to
          just keep whatever the final epoch produced (the old behavior).

        The trained continuous parameters are kept on this instance (usable
        via `predict_continuous` / `evaluate_continuous`, see below) whether
        or not you discretize. If `commit=True` (the default), every
        continuous parameter is *also* discretized (argmax / sign / rounding)
        and written directly back into this instance's own `deflectors_list`,
        `alphabet_directions` and `alphabet_start_positions` attributes, so
        every other method on this class (compute_input, ricochet, clone,
        Mutate, display_deflectors, ...) keeps working exactly as before,
        just with gradient-trained values in place of the old ones. Pass
        `commit=False` to leave those discrete attributes untouched and only
        keep the trained continuous version (e.g. if you want to compare
        continuous vs. discrete behavior, or keep iterating with
        `train_continuous` again before ever committing to a discrete brain).

        Returns `(loss_history, accuracy_history)`, both per-epoch lists of
        floats (one entry per full pass over the data, averaged across that
        epoch's batches) -- these always reflect what actually happened each
        epoch, even if `restore_best=True` rewound the instance's live
        parameters to an earlier, better checkpoint afterward. `accuracy_history` is a cheap "token accuracy" over
        *every* trajectory position seen during the epoch (aligned or blank):
        it checks whether that position's predicted class -- its top symbol if
        its emit-probability is >= 0.5, else "blank" -- matches the target
        class, and reports the fraction that do. It's computed for free from
        the same numbers already used for the loss -- no extra simulation is
        run -- so while it now also reflects whether leading positions are
        correctly suppressed, it's still not the same as full-sequence
        exact-match correctness on the real, discrete brain (see
        `evaluate_continuous` / `evaluate_performance` for that).
        """
def generate_test_cases(amount, number, indiv=True):
  #x = TokenizeInput(X, alphabet)
  test_cases = []
  if indiv == True:
    test_cases = [(f'1+{i}', f"{random.randint(0, 1000)}") for i in range(amount)]
  else:

    for i in range(amount):
      a = []
      if i != len(x)-1: b = x[i+1]
      for j in range(i+1):
        a.append(x[j])
        if len(a) > 25:
            a.pop(0)
      test_cases.append((''.join(a), b))

  return test_cases

## Fitness evaluation

Fitness evaluation module for Ball_Trap_Brain genetic algorithm.

This module contains functions to evaluate the performance of individual brains
and entire populations against test cases using string similarity metrics.

In [ ]:

import difflib
import math


def evaluate_performance(input_val, answer, brain, ratio=False):

    output = brain.compute_input(input_val)
    if ratio:
        return difflib.SequenceMatcher(None, ''.join(output), answer).ratio()
    else:
      return 1 if ''.join(output) == answer else 0


import concurrent.futures

def eval_single_brain(args):
    individual, test_cases, ratio = args
    avg_fitness = sum(
        evaluate_performance(inp, ans, individual, ratio) for inp, ans in test_cases
    ) / len(test_cases)
    return avg_fitness

def evaluate_population_parallel(population, test_cases, ratio=False):
    # Package arguments for the worker pool
    args = [(ind, test_cases, ratio) for ind in population]

    # Run across available CPU cores
    with concurrent.futures.ProcessPoolExecutor() as executor:
        results = list(executor.map(eval_single_brain, args))

    best_fitness = max(results)
    best_brain = population[results.index(best_fitness)]
    return best_fitness, best_brain

In [ ]:
def get_continuous_combinations(s):
    result = []
    n = len(s)
    for i in range(n):
        for j in range(i + 1, i + 5):
            result.append(s[i:j])
    return result

## Genetic algorithm (`main`)

Genetic algorithm to evolve Ball_Trap_Brain instances to produce desired outputs.

This module runs a genetic algorithm that evolves a population of brains over
multiple generations. Each brain is evaluated against test cases, and the best
performer is mutated to create the next generation.

In [ ]:

string = "This module runs a genetic algorithm that evolves a population of brains over multiple generations. Each brain is evaluated against test cases, and the best performer is mutated to create the next generation."

import copy
import difflib
import random
import shelve

# Test cases: pairs of (input_string, expected_output_string) used for fitness evaluation



alphabet = get_continuous_combinations(x)  # Define the alphabet used in the brains
output = list("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ`_*(),.:[]/|\()-<>= ")

def main(mini_batch, alphabet, generations=500, pop_size=100, brain=Ball_Trap_Brain(1000, 4, alphabet, output, 2), ratio=False):
    """
    Run the genetic algorithm for evolving Ball_Trap_Brain instances.

    Returns:
        Ball_Trap_Brain: The best brain found after evolution.
    """
    # Evolution hyperparameters
    population_size = pop_size

    # Initialize population with random brains
    # Parameters: 10 deflectors, 2 bounce energy, alphabet of 6 characters

    population = [brain.clone().Mutate(1.0, 0.025) for _ in range(population_size)]
    population.append(brain)
    best_individual = None
    mini_batch_0 = copy.copy(mini_batch)
    #a = [mini_batch_0.pop(0) for i in range(10)]
    highest_score = -1
    # Run evolution loop
    for generation in range(generations):

        # Evaluate all brains and find the best performer
        best_fitness, best_individual = evaluate_population_parallel(population,mini_batch, ratio)



        # Check for perfect solution
        if best_fitness >= 1:
          print(f"Generation {generation}: Solution found!")
          return best_individual
          #for i in range(10):
            #a.append(mini_batch_0.pop(0))


        # Create next generation by mutating the best brain multiple times
        # New code: passes best_fitness, individual_probability, base_probability
        new_population = [best_individual.clone().Mutate(1.0, 0.025) for _ in range(population_size)]
        population = new_population
        population.append(best_individual)
        print(f"Generation {generation}: Best fitness = {best_fitness}")
    return best_individual

training_1 = generate_test_cases(amount=100, number='1', indiv=True)

<>:13: SyntaxWarning: invalid escape sequence '\('
<>:13: SyntaxWarning: invalid escape sequence '\('
/tmp/ipykernel_2472/1267424427.py:13: SyntaxWarning: invalid escape sequence '\('
  output = list("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ`_*(),.:[]/|\()-<>= ")


In [ ]:
def strip_continuous(brain):
    for attr in ['_c_dest_logits', '_c_hole_logit', '_c_symbol_logits',
                 '_c_bounce_val', '_c_direction_logits', '_c_start_logits', '_c_meta']:
        if hasattr(brain, attr):
            delattr(brain, attr)
    return brain

## `Council_Brain` & council training

In [ ]:

from collections import Counter

class Council_Brain:
    def __init__(self, brains):
        """
        Initializes the council with a list of separately trained brains.
        """
        self.brains = brains

    def compute_input(self, input_val):
        """
        Each brain processes the input separately.
        The council votes on the final answer by finding the most common output.
        """
        votes = []
        for brain in self.brains:
            # Each brain outputs a list of characters; join them to form the string
            output_str = ''.join(brain.compute_input(input_val))

            votes.append(output_str)

        # Handle empty cases securely
        if not votes:
            return []

        # Tally the votes and find the majority
        vote_counts = Counter(votes)
        majority_vote = vote_counts.most_common(1)[0][0]

        # Return as a list of characters to perfectly match the original Ball_Trap_Brain format
        return list(majority_vote)

def train_council(num_brains, training_data, alphabet, output_format, generations=50, pop_size=100):
    """
    Trains a council where each brain is trained separately using the genetic algorithm.
    """
    trained_brains = []

    for i in range(num_brains):
        print(f"\n=====================================")
        print(f" Training Council Brain {i+1} / {num_brains}")
        print(f"=====================================")
        base_brain = Ball_Trap_Brain(50, 10, alphabet, output, 2)
        # Initialize a fresh brain for this specific council member
        #base_brain = Ball_Trap_Brain(100, 10, alphabet, output_format, 2)

        # Train separately using the existing main GA loop
        for j in range(1):
          print(f'---{j}---')
          base_brain.train_continuous(
              random.sample(training_data,10),
              epochs=20,
              steps=generations,
              lr_start=0.4,
              lr_end=0.1,      # cosine-decays alongside temp/index_temp for late-training stability
              grad_clip_norm=1.0, # caps how large a single overshoot can be, regardless of temperature
              restore_best=False,  # roll back to the best checkpoint seen, in case of a late loss spike
              batch_size=5,
              warm_start=True,
              commit=True,   # keep the discrete brain untouched; only train the continuous version
              verbose=True,
              temp_start=2.5,
              temp_end=1,       # anneals the learnable-logit softmaxes (dest/symbol/hole/start)
              index_temp_start=2.5,
              index_temp_end=1,   # anneals the non-learnable start-index kernel much further down,
                                      # since nothing else can sharpen it -- see train_continuous's docstring
              index_temp_power=2.0,  # spends more epochs in the middle-to-low range before tightening
                                      # to its sharpest, least-forgiving point, instead of a flat linear cool
              blank_loss_weight=1,
          )
        strip_continuous(base_brain)
        trained_brains.append(base_brain)

    return Council_Brain(trained_brains)

## Continuous relaxation training — demo

Trains a brain with gradient descent (requires `torch`; installed via `pip install torch` if missing) with `commit=False`, so the discrete brain is left completely untouched, and answers questions directly with the continuous version via `predict_continuous`. Then calls `commit_continuous()` and compares against the real, discrete `compute_input`. The committed brain now also receives the trained Q/K/V head, so it is no longer using a different hand-written attention function.


In [ ]:
# Continuous -> discrete attention consistency check
def check_attention_commit_consistency(brain, input_str):
    """Compare the learned continuous attention context with the committed
    discrete attention context using the same hard token choices."""
    import torch
    if not hasattr(brain, '_c_meta') or not hasattr(brain, '_discrete_q_proj'):
        raise RuntimeError('Train and commit the continuous brain first.')
    tokens = TokenizeInput(input_str, brain.alphabet)
    idx = [brain.alphabet.index(t) for t in tokens if t in brain.alphabet]
    dirs_h = [brain.alphabet_directions[i] for i in idx]
    starts_h = [brain.alphabet_start_positions[i] for i in idx]
    if len(idx) <= 1:
        return {'max_abs_context_error': 0.0, 'continuous_context': [0.0]*len(idx), 'discrete_context': [0]*len(idx)}
    with torch.no_grad():
        dev = brain._c_q_proj.device
        dirs = torch.tensor(dirs_h, dtype=torch.float32, device=dev).unsqueeze(0)
        starts = torch.tensor(starts_h, dtype=torch.float32, device=dev).unsqueeze(0)
        valid = torch.ones((1, len(idx)), dtype=torch.bool, device=dev)
        cont = (torch.tanh(brain._c_attn_gain) * brain._c_attention(dirs, starts, valid)).squeeze(0).cpu().tolist()
    disc = [brain._single_head_attention_discrete(dirs_h, starts_h, i) for i in range(len(idx))]
    err = max(abs(float(a) - float(b)) for a,b in zip(cont, disc)) if disc else 0.0
    return {'max_abs_context_error': err, 'continuous_context': cont, 'discrete_context': disc}


In [ ]:
# Attention diagnostics
def inspect_attention(brain, input_str):
    import torch
    if not hasattr(brain, '_c_meta'):
        raise RuntimeError('Train the continuous brain first.')
    tokens = TokenizeInput(input_str, brain.alphabet)
    idx = [brain.alphabet.index(t) for t in tokens if t in brain.alphabet]
    if not idx: return []
    with torch.no_grad():
        dirs = brain._c_meta['direction_values'][brain._c_direction_logits.argmax(1)[idx]]
        starts = brain._c_meta['positions'][brain._c_start_logits.argmax(1)[idx]]
        valid = torch.ones((1,len(idx)),dtype=torch.bool,device=dirs.device)
        ctx = brain._c_attention(dirs.unsqueeze(0),starts.unsqueeze(0),valid)
        gain = torch.tanh(brain._c_attn_gain).item()
    return {'tokens':tokens,'context':ctx.squeeze(0).cpu().tolist(),'gain':gain}

# Example after training: print(inspect_attention(demo_brain, 'alpha beta gamma'))


In [ ]:

# !pip install torch --quiet   # uncomment if torch isn't already installed

import random
#random.seed(0)

demo_brain = Ball_Trap_Brain(100, 10, alphabet, output, 1, direction_range=20)
cases = generate_test_cases(amount=100, number='1', indiv=False)  # '1+i' -> str(1+i)
random.shuffle(cases)
share = 0.8
demo_cases = cases[:int(share*len(cases))]
demo_test = cases[int(share*len(cases)):]
for i in range(1):
  loss_history, acc_history = demo_brain.train_continuous(
      demo_cases,
      epochs=100,
      steps=10000,
      lr_start=0.8,
      lr_end=0.1,      # cosine-decays alongside temp/index_temp for late-training stability
      grad_clip_norm=1, # caps how large a single overshoot can be, regardless of temperature
      restore_best=False,  # roll back to the best checkpoint seen, in case of a late loss spike
      batch_size=25,
      warm_start=True,
      commit=False,   # keep the discrete brain untouched; only train the continuous version
      verbose=True,
      temp_start=2.5,
      temp_end=1,       # anneals the learnable-logit softmaxes (dest/symbol/hole/start)
      index_temp_start=2.5,
      index_temp_end=1,   # anneals the non-learnable start-index kernel much further down,
                              # since nothing else can sharpen it -- see train_continuous's docstring
      index_temp_power=2.0,  # spends more epochs in the middle-to-low range before tightening
                              # to its sharpest, least-forgiving point, instead of a flat linear cool
      blank_loss_weight=1
  )
  print('token_acc[0] ->', acc_history[0], ' token_acc[-1] ->', acc_history[-1])

  print()
  print('=== CONTINUOUS predictions ===')
  # for inp, ans in demo_cases[:20]:
  #      pred = demo_brain.predict_continuous(inp)
  #      print(inp, '->', repr(pred), ' (target', ans, ') match:', pred == ans)
  cont_exact = demo_brain.evaluate_continuous(demo_cases, ratio=False)
  cont_ratio = demo_brain.evaluate_continuous(demo_cases, ratio=True)
  print('continuous exact-match:', cont_exact, ' ratio:', cont_ratio)
  print('continuous test match:', demo_brain.evaluate_continuous(demo_test, ratio=False))


demo_brain.commit_continuous()
print('attention commit check:', check_attention_commit_consistency(demo_brain, demo_cases[0][0]))
print()
print('=== DISCRETE predictions (after commit_continuous) ===')
# for inp, ans in demo_cases[:10]:
#     pred = ''.join(demo_brain.compute_input(inp))
#     print(inp, '->', repr(pred), ' (target', ans, ') match:', pred == ans)
disc_exact = sum(evaluate_performance(i, a, demo_brain, ratio=False) for i, a in demo_cases) / len(demo_cases)
disc_ratio = sum(evaluate_performance(i, a, demo_brain, ratio=True) for i, a in demo_cases) / len(demo_cases)
print('discrete exact-match:', disc_exact, ' ratio:', disc_ratio)
print('discrete test match:', sum(evaluate_performance(i, a, demo_brain, ratio=False) for i, a in demo_test) / len(demo_test))


[continuous] step 0: loss=5.8105 token_acc=0.000 fire_acc=0.000 temp=2.50 index_temp=2.500 lr=0.80000 batch_size=25 batches=1
[continuous] step 500: loss=1.2568 token_acc=0.903 fire_acc=0.680 temp=2.42 index_temp=2.496 lr=0.79569 batch_size=25 batches=1
[continuous] step 1000: loss=1.5414 token_acc=0.864 fire_acc=0.520 temp=2.35 index_temp=2.485 lr=0.78287 batch_size=25 batches=1
[continuous] step 1500: loss=1.5171 token_acc=0.850 fire_acc=0.480 temp=2.27 index_temp=2.466 lr=0.76184 batch_size=25 batches=1
[continuous] step 2000: loss=1.2558 token_acc=0.834 fire_acc=0.609 temp=2.20 index_temp=2.440 lr=0.73314 batch_size=25 batches=1
[continuous] step 2500: loss=1.1853 token_acc=0.871 fire_acc=0.667 temp=2.12 index_temp=2.406 lr=0.69747 batch_size=25 batches=1
[continuous] step 3000: loss=1.2037 token_acc=0.849 fire_acc=0.750 temp=2.05 index_temp=2.365 lr=0.65570 batch_size=25 batches=1
[continuous] step 3500: loss=1.2710 token_acc=0.865 fire_acc=0.625 temp=1.97 index_temp=2.316 lr=0.60

## Full council training run (optional, slow)

This reproduces the original script's `if __name__ == '__main__':` block: it trains a 50-brain council with the genetic algorithm (400 generations each) and evaluates it. This can take a long time, so it's guarded behind `RUN_FULL_TRAINING`.

In [ ]:

RUN_FULL_TRAINING = False # set to True to actually run the full council training

if RUN_FULL_TRAINING:
    # Generate the test cases using the existing generation function
    test_cases = generate_test_cases(amount=15, number='1', indiv=False)
    #random.shuffle(test_cases)
    split = 1
    # Split the dataset
    training_0 = test_cases[:int(split * len(test_cases))]
    training_test = test_cases[int(split * len(test_cases)):]


    # Define council parameters
    COUNCIL_SIZE = 5      # Number of voting brains in the council
    GENERATIONS = 200     # Generations per brain
    POP_SIZE = 100        # Population size per generation

    # Train the council (each brain is trained entirely separately)
    print(f"Initializing Council Training with {COUNCIL_SIZE} brains...")
    council = train_council(
        num_brains=COUNCIL_SIZE,
        training_data=training_0,
        alphabet=alphabet,
        output_format=output,
        generations=GENERATIONS,
        pop_size=POP_SIZE
    )



    # Evaluate the entire council's performance against the testing set
    # Note: evaluate_population_parallel relies on a list of brains, so we wrap the council in a list
    print("\nEvaluating Council performance on test batch...")
    council_fitness, _ = (0, 0)#evaluate_population_parallel([council], training_test, False)
    council_fitness_training, _ = evaluate_population_parallel([council], training_0, False)
    print(f"Overall Council Fitness: {council_fitness}, Council training fitness: {council_fitness_training}")